In [ ]:
from pathlib import Path
import xarray as xr
import shnitsel as st
import numpy as np

from shnitsel.data.tree.node import TreeNode

old_data_folder = Path("../test_data/shnitsel/old_sd/")
new_data_folder = Path("../test_data/shnitsel/new_sd/")

def convert_old_data(ds:TreeNode|xr.Dataset):
    if isinstance(ds, TreeNode):
        return ds.map_data(convert_old_data)

    if 'forces' in ds:
        return ds.assign(forces=-ds.forces)

for pth in old_data_folder.iterdir():
    filename = pth.name
    compound_name = filename.split('_')[0]
    new_path = new_data_folder/filename
    if not new_path.exists() and pth.is_file() and pth.exists() and not filename.startswith("."):
        print("Reading: ",pth)
        data = st.read(pth)
        
        if isinstance(data, TreeNode):
            data = data.set_compound_info(compound_name)
            display(next(iter(data[compound_name].children.items())))
        print("Converting: ",pth)
        converted_data = convert_old_data(data)
        display(converted_data)
        print("Writing to: ",new_path)
        st.write_shnitsel_file(converted_data, savepath=new_path, output_engine='h5netcdf')